# Phase C — 10 Attention Mechanisms Investigation (E06–E15) on Winning PDN-S Backbone

Evaluates **10 attention mechanisms** inserted at the exact same point in the Phase B winning **`PDN-S` Student** (`conv3` output: `(B, 256, 59, 59)` → `(B, 256, 59, 59)`), under the exact same controlled protocol as `E01_PUAD_200ep` (`89.64%` Combined Mean / `84.07%` Pooled Raw):

- **E06**: SE (Squeeze-and-Excitation)
- **E07**: ECA (Efficient Channel Attention)
- **E08**: CBAM (Convolutional Block Attention Module)
- **E09**: Coordinate Attention (CoordAtt)
- **E10**: Triplet Attention
- **E11**: SimAM (Parameter-Free 3D Energy)
- **E12**: GCT (Gated Channel Transformation)
- **E13**: BAM (Bottleneck Attention Module)
- **E14**: Non-Local Self-Attention
- **E15**: Criss-Cross Attention (R=2)

All runs auto-save every epoch to Google Drive and auto-skip if already completed, so you can run all 10 across 1 or multiple Colab sessions effortlessly.

In [1]:
#@title 1. Section 3 — Hardware, Runtime, RAM & Disk Audit
import os, shutil, sys, torch, psutil
print('=== COLAB SESSION AUDIT (PHASE C) ===')
print('Python        :', sys.version.split()[0])
print('PyTorch       :', torch.__version__, '| CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'No GPU assigned!'
props = torch.cuda.get_device_properties(0)
print(f'GPU Model     : {props.name} (CC {props.major}.{props.minor}) | VRAM: {props.total_memory/(1024**3):.2f} GiB | Native BF16: {props.major >= 8 and torch.cuda.is_bf16_supported()}')
vm = psutil.virtual_memory()
print(f'System RAM    : {vm.available/(1024**3):.1f} GB free / {vm.total/(1024**3):.1f} GB total')
disk = shutil.disk_usage('/content')
print(f'Local SSD     : {disk.free/(1024**3):.1f} GB free / {disk.total/(1024**3):.1f} GB total')

=== COLAB SESSION AUDIT (PHASE C) ===
Python        : 3.13.16
PyTorch       : 2.14.1+cu130 | CUDA: 13.0
GPU Model     : NVIDIA A100-SXM4-40GB (CC 8.0) | VRAM: 39.49 GiB | Native BF16: True
System RAM    : 81.1 GB free / 83.5 GB total
Local SSD     : 235.6 GB free / 235.7 GB total


In [2]:
#@title 2. Mount Drive, Pull Latest Code & Ensure Local SSD Datasets
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/DL_Endsem_Team11'
DRIVE_LOCO_TAR    = f'{DRIVE_PROJECT_DIR}/data/mvtec_loco_anomaly_detection.tar.xz'
DRIVE_IMNET_ZIP   = f'{DRIVE_PROJECT_DIR}/data/imagenet_penalty_10k.zip'
DRIVE_TEACHER_PT  = f'{DRIVE_PROJECT_DIR}/weights/teacher_pdn_s.pt'
DRIVE_RESULTS_DIR = f'{DRIVE_PROJECT_DIR}/runs'
GITHUB_REPO_URL   = 'https://github.com/Chimera418/dl-endsem-logical-ad.git'

REPO_DIR = '/content/Endsem'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO_URL, REPO_DIR], check=True)

!pip install -q timm thop einops pyyaml psutil tqdm
%cd /content/Endsem

LOCAL_LOCO = '/content/data/mvtec_loco_anomaly_detection'
LOCAL_IMNET = '/content/data/imagenet_penalty'
os.makedirs('/content/data', exist_ok=True)
if not os.path.isdir(os.path.join(LOCAL_LOCO, 'breakfast_box')):
    os.makedirs(LOCAL_LOCO, exist_ok=True)
    !tar -xf "$DRIVE_LOCO_TAR" -C "$LOCAL_LOCO"
if not os.path.isdir(LOCAL_IMNET):
    !unzip -q "$DRIVE_IMNET_ZIP" -d /content/data/

os.environ['ENDSEM_LOCO_ROOT'] = LOCAL_LOCO
os.environ['ENDSEM_IMAGENET_ROOT'] = LOCAL_IMNET
os.environ['ENDSEM_TEACHER_PATH'] = DRIVE_TEACHER_PT
!python scripts/verify_env.py

Mounted at /content/drive
/content/Endsem
SECTION 3 — ENVIRONMENT & DATASET VERIFICATION AUDIT
1. CUDA Available    : True
2. GPU Model         : NVIDIA A100-SXM4-40GB (CC 8.0, BF16=True)
3. GPU Memory        : 39.49 GiB (40441 MiB)
4. PyTorch / CUDA    : PyTorch 2.14.1+cu130 | CUDA 13.0 | cuDNN 92800
   Python / Packages : Python 3.13.16 | timm=1.0.31 | thop=0.1.1 | mamba_ssm=None
5. System RAM        : 80.27 GB free / 83.47 GB total (3.8% used)
6. Disk Space        : 223.53 GB free / 235.68 GB total (/content/Endsem)
------------------------------------------------------------------------------
7. Dataset & Weights Accessibility:
   MVTec LOCO AD     : OK (/content/data/mvtec_loco_anomaly_detection)
     train=1778 | val=305 | test_good=575 | test_logical=561 | test_structural=432 (EXCLUDED)
   ImageNet Penalty  : OK (10,000 images at /content/data/imagenet_penalty)
   PDN-S Teacher     : OK (/content/drive/MyDrive/DL_Endsem_Team11/weights/teacher_pdn_s.pt)


In [3]:
#@title 3. Batch 1 — Channel & Hybrid Attentions (E06 SE, E07 ECA, E08 CBAM, E09 CoordAtt, E10 Triplet)
!python scripts/run_experiment.py --config configs/E06_se.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E07_eca.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E08_cbam.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E09_coordatt.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E10_triplet.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"

RUN: E06_SE | mode=train | device=cuda | seed=42
Dataset audit OK: train=1778 | val=305 | test_good=575 | test_logical=561 | structural_excluded=432
Loading pooled train/good and validation/good (img_size=256, cache_in_ram=True)...
  Penalty corpus ready: 10,000 images (batch_size=8)
  Computing streaming teacher channel normalization stats on pooled train/good...
  Starting training: epochs=200, batch_size=8, amp=torch.bfloat16, trainable_params=5,372,176, frozen_params=2,694,144
  Epoch   1/200 | train=11.0929 | val=7.8929 | lr=3.0e-04 | 30.9s <-- BEST
  Epoch   2/200 | train=6.7659 | val=5.9160 | lr=3.0e-04 | 16.6s <-- BEST
  Epoch   3/200 | train=5.3268 | val=4.8441 | lr=3.0e-04 | 16.4s <-- BEST
  Epoch   4/200 | train=4.5698 | val=4.2517 | lr=3.0e-04 | 16.6s <-- BEST
  Epoch   5/200 | train=4.1360 | val=4.0353 | lr=3.0e-04 | 16.2s <-- BEST
  Epoch   6/200 | train=3.8167 | val=3.6944 | lr=3.0e-04 | 16.4s <-- BEST
  Epoch   7/200 | train=3.6033 | val=3.5083 | lr=3.0e-04 | 16.6s <-- 

In [6]:
import os, subprocess, time, pathlib

os.chdir("/content/Endsem")
DRIVE_RUNS = "/content/drive/MyDrive/DL_Endsem_Team11/runs"
LOG_DIR = pathlib.Path("/content/Endsem/parallel_logs")
LOG_DIR.mkdir(exist_ok=True)

CONFIGS = [
    "configs/E07_eca.yaml",
    "configs/E08_cbam.yaml",
    "configs/E09_coordatt.yaml",
    "configs/E10_triplet.yaml",
]

env = os.environ.copy()
env["OMP_NUM_THREADS"] = "2"
env["PYTHONUNBUFFERED"] = "1"

procs = {}
for cfg in CONFIGS:
    name = pathlib.Path(cfg).stem
    log_path = LOG_DIR / f"{name}.log"
    f = open(log_path, "w")
    p = subprocess.Popen(
        ["python", "scripts/run_experiment.py", "--config", cfg, "--mode", "train", "--drive-sync-dir", DRIVE_RUNS],
        stdout=f, stderr=subprocess.STDOUT, env=env,
    )
    procs[name] = (p, f, log_path)
print(f"Launched {len(procs)} parallel runs on A100: {list(procs.keys())}\n")

while True:
    alive = 0
    print(f"--- Status @ {time.strftime('%H:%M:%S')} ---")
    for name, (p, f, log_path) in procs.items():
        rc = p.poll()
        status = "RUNNING" if rc is None else f"DONE(rc={rc})"
        if rc is None:
            alive += 1
        lines = [l.strip() for l in log_path.read_text(errors="ignore").splitlines() if l.strip()]
        last = lines[-1] if lines else "(starting...)"
        print(f"  [{status:10s}] {name:16s} | {last[:95]}")
    if alive == 0:
        break
    time.sleep(60)

for name, (p, f, log_path) in procs.items():
    f.close()
    print(f"\n{'='*70}\nFINAL TAIL: {name} (rc={p.returncode})\n{'='*70}")
    print("\n".join(log_path.read_text(errors="ignore").splitlines()[-22:]))

Launched 4 parallel runs on A100: ['E07_eca', 'E08_cbam', 'E09_coordatt', 'E10_triplet']

--- Status @ 07:20:18 ---
  [RUNNING   ] E07_eca          | (starting...)
  [RUNNING   ] E08_cbam         | (starting...)
  [RUNNING   ] E09_coordatt     | (starting...)
  [RUNNING   ] E10_triplet      | (starting...)
--- Status @ 07:21:18 ---
  [RUNNING   ] E07_eca          | Loading pooled train/good and validation/good (img_size=256, cache_in_ram=True)...
  [RUNNING   ] E08_cbam         | Loading pooled train/good and validation/good (img_size=256, cache_in_ram=True)...
  [RUNNING   ] E09_coordatt     | Loading pooled train/good and validation/good (img_size=256, cache_in_ram=True)...
  [RUNNING   ] E10_triplet      | Loading pooled train/good and validation/good (img_size=256, cache_in_ram=True)...
--- Status @ 07:22:18 ---
  [RUNNING   ] E07_eca          | Loading pooled train/good and validation/good (img_size=256, cache_in_ram=True)...
  [RUNNING   ] E08_cbam         | Loading pooled train/

In [ ]:
#@title 4. Batch 2 — Energy, Gated, Dilated & Global Attentions (E11 SimAM, E12 GCT, E13 BAM, E14 NonLocal, E15 CrissCross)
!python scripts/run_experiment.py --config configs/E11_simam.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E12_gct.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E13_bam.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E14_nonlocal.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E15_crisscross.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"

In [ ]:
import os, subprocess, time, pathlib

os.chdir("/content/Endsem")
DRIVE_RUNS = "/content/drive/MyDrive/DL_Endsem_Team11/runs"
LOG_DIR = pathlib.Path("/content/Endsem/parallel_logs")
LOG_DIR.mkdir(exist_ok=True)

CONFIGS = [
    "configs/E11_simam.yaml",
    "configs/E12_gct.yaml",
    "configs/E13_bam.yaml",
    "configs/E14_nonlocal.yaml",
    "configs/E15_crisscross.yaml",
]

env = os.environ.copy()
env["OMP_NUM_THREADS"] = "2"
env["PYTHONUNBUFFERED"] = "1"

procs = {}
for cfg in CONFIGS:
    name = pathlib.Path(cfg).stem
    log_path = LOG_DIR / f"{name}.log"
    f = open(log_path, "w")
    p = subprocess.Popen(
        ["python", "scripts/run_experiment.py", "--config", cfg, "--mode", "train", "--drive-sync-dir", DRIVE_RUNS],
        stdout=f, stderr=subprocess.STDOUT, env=env,
    )
    procs[name] = (p, f, log_path)
print(f"Launched {len(procs)} parallel runs on A100: {list(procs.keys())}\n")

while True:
    alive = 0
    print(f"--- Status @ {time.strftime('%H:%M:%S')} ---")
    for name, (p, f, log_path) in procs.items():
        rc = p.poll()
        status = "RUNNING" if rc is None else f"DONE(rc={rc})"
        if rc is None:
            alive += 1
        lines = [l.strip() for l in log_path.read_text(errors="ignore").splitlines() if l.strip()]
        last = lines[-1] if lines else "(starting...)"
        print(f"  [{status:10s}] {name:16s} | {last[:95]}")
    if alive == 0:
        break
    time.sleep(60)

for name, (p, f, log_path) in procs.items():
    f.close()
    print(f"\n{'='*70}\nFINAL TAIL: {name} (rc={p.returncode})\n{'='*70}")
    print("\n".join(log_path.read_text(errors="ignore").splitlines()[-22:]))

In [ ]:
for cfg in configs/E06_se.yaml configs/E07_eca.yaml configs/E08_cbam.yaml configs/E09_coordatt.yaml configs/E10_triplet.yaml configs/E11_simam.yaml configs/E12_gct.yaml configs/E13_bam.yaml configs/E14_nonlocal.yaml configs/E15_crisscross.yaml; do
  python scripts/run_experiment.py --config "$cfg" --mode eval --drive-sync-dir "$DRIVE_RUNS"
done

In [ ]:
#@title 5. Aggregate & Compare All Experiments (E01–E15 Master Table)
!python scripts/aggregate_results.py --dirs experiments "$DRIVE_RESULTS_DIR" --out-dir "$DRIVE_RESULTS_DIR/phaseC_summary"